# Merging tables safely

Python stage, lesson P15. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** state how many partners a key may have and write it in `validate` before merging; choose `inner`, `left` or `outer` and predict the row count first; use `indicator` to see where each row came from; spot a many-to-many merge by its row count.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")
# The order lines with their price and amount, as built in lesson P13.
price_of = dict(zip(products["product_id"], products["unit_price_usd"]))
lines = order_items.assign(price=order_items["product_id"].map(price_of))
lines = lines.assign(amount=lines["quantity"] * lines["price"] - lines["discount_usd"])

## Unit 1: merge, and validate what you expect

**Predict first.** each order has one store. How many rows and columns will `orders_stores` have? `orders` has 6 columns, `stores` has 4, and the key is in both.

In [ ]:
orders_stores = orders.merge(stores, on="store_id", validate="many_to_one")
print(orders.shape, stores.shape, orders_stores.shape)

**Predict first.** `one_to_one` says that every key is unique on both sides, but a store appears many times in `orders`. Does `merge` return a smaller table, print a warning, or stop?

In [ ]:
try:
    orders.merge(stores, on="store_id", validate="one_to_one")
except pd.errors.MergeError as error:
    print(error.args[0].split("\n")[0])

## Unit 2: how: what happens to a row with no partner

**Predict first.** `orders` holds one walk-in order with no customer id, and some customers have never ordered. Will `inner` have 1,800 rows? Which of `inner` and `left` has more rows?

In [ ]:
inner = customers.merge(orders, on="customer_id", how="inner", validate="one_to_many")
left = customers.merge(orders, on="customer_id", how="left", validate="one_to_many")
print(len(customers), len(orders))
print(len(inner), len(left))

## Unit 3: indicator: where each row came from

**Predict first.** how many rows of `audit` are `both`, how many `left_only` and how many `right_only`?

In [ ]:
audit = customers.merge(orders, on="customer_id", how="outer", validate="one_to_many", indicator=True)
print(audit["_merge"].value_counts())
print(len(audit))

## Unit 4: many to many: the rows that multiply

**Predict first.** each Latte line meets both Latte rows of `promo`, and each other line finds one row or none. How many rows will `with_promo` have?

In [ ]:
promo = pd.DataFrame({"product_id": [3, 3, 5], "promo": ["student", "staff", "student"]})
with_promo = order_items.merge(promo, on="product_id", how="left", validate="many_to_many")
print(len(order_items), len(with_promo))

**Predict first.** do the two totals agree? The second is the control total of the order lines.

In [ ]:
priced = lines.merge(promo, on="product_id", how="left", validate="many_to_many")
print(priced["amount"].sum(), lines["amount"].sum())

## Unit 5: Check: a merge instead of map

**Predict first.** is the row count unchanged, and what do the amounts add up to?

In [ ]:
with_price = order_items.merge(products[["product_id", "unit_price_usd"]], on="product_id", how="left", validate="many_to_one")
amount = with_price["quantity"] * with_price["unit_price_usd"] - with_price["discount_usd"]
print(len(with_price), len(order_items))
print(amount.sum())

## Finish it

### Exercise 1: Finish it

**Task:** How many orders were placed by customers who live in Oakland? Replace the first `None` with `orders` merged with `customer_city`: choose how the orders without a customer are kept, and write the promise the keys make. Then set `answer_1` to the number of rows of `with_city` whose city is Oakland.

**Expected:** one whole number.

In [ ]:
customer_city = customers[["customer_id", "city"]]
with_city = None  # replace None: orders merged with customer_city on customer_id, with how and validate chosen
answer_1 = None  # then set answer_1 to the number of rows of with_city whose city is "Oakland"

In [ ]:
_ = check("Exercise 1", "answer_1", 689, traps={44: "44 is the number of Oakland customers. The question asks for orders, and a customer has many of them: count the rows of with_city."})

<details>
<summary>Hint</summary>

Each order has one customer, so the customers are the right side and each key is unique there. Unit 2 says what `how` does with an order that has no customer.

</details>

<details>
<summary>Solution</summary>

```python
customer_city = customers[["customer_id", "city"]]
with_city = orders.merge(customer_city, on="customer_id", how="left", validate="many_to_one")
answer_1 = (with_city["city"] == "Oakland").sum()
```

The answer is 689: with `how="left"` all 1,800 rows of `orders` are kept, and 689 of them belong to Oakland customers. The 44 Oakland customers have many orders each, so counting customers gives 44, and the check cell says so.

</details>

## On your own

### Exercise 2: On your own

**Task:** How many order lines were rung up on orders paid in cash? Before you start, say in one sentence how many rows the merged table should have, and why. Merge the order lines with the payment method of their order, with a promise in `validate`, then count. Write the code in the empty cell below and set `answer_2`.

**Expected:** one whole number.

In [ ]:
_ = check("Exercise 2", "answer_2", 453, traps={253: "253 is the number of cash orders. An order can have several lines, so the lines are more: merge the lines with their orders and count the rows."})

<details>
<summary>Hint</summary>

The table of lines has one row for each line, and each line belongs to one order. Which table holds the payment method, and which side repeats its keys?

</details>

<details>
<summary>Solution</summary>

```python
line_orders = order_items.merge(orders[["order_id", "payment_method"]], on="order_id", how="left", validate="many_to_one")
answer_2 = (line_orders["payment_method"] == "cash").sum()
```

The answer is 453: the 3,287 lines keep their rows (many lines, one order each), and 453 of them belong to cash orders. Counting `orders` alone gives 253, the number of cash orders, and the check cell says so.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** Spot the flaw. A colleague attaches this week's specials to the order lines and counts the lines that are on special. The Blueberry Muffin, product 11, is listed twice because a 2 for 1 offer was canceled in March and never removed. Fix the lines so that every line is counted once, with a promise in `validate`, then set `answer_3` to the number of lines on special.

**Expected:** one whole number.

In [ ]:
special = pd.DataFrame({"product_id": [10, 11, 11], "special": ["half price", "half price", "2 for 1"]})
on_special = order_items.merge(special, on="product_id", how="left")  # the colleague's line
answer_3 = None  # fix the lines above, then set answer_3 = the number of rows of on_special whose special is not blank

In [ ]:
_ = check("Exercise 3", "answer_3", 531, traps={755: "755 counts each Blueberry Muffin line twice. Remove the canceled 2 for 1 row from special, and add validate so that the merge stops if a key repeats."})

<details>
<summary>Hint</summary>

Compare `len(on_special)` with `len(order_items)`. A left merge with a table whose keys are unique keeps the rows of the left table.

</details>

<details>
<summary>Solution</summary>

```python
special = pd.DataFrame({"product_id": [10, 11, 11], "special": ["half price", "half price", "2 for 1"]})
special = special[special["special"] != "2 for 1"]
on_special = order_items.merge(special, on="product_id", how="left", validate="many_to_one")
answer_3 = on_special["special"].notna().sum()
```

The answer is 531: 307 lines of product 10 and 224 of product 11, each counted once. The colleague's merge gave 755, because the 224 Blueberry Muffin lines met two rows of `special`; the check cell says so. With `validate="many_to_one"` the colleague's merge would have stopped.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")